In [1]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import mysql.connector

In [ ]:
# EXTRACT

df = pd.read_csv(
    r"D:\Data Engineer\Sample_Superstore.csv",
    encoding='latin1',
    on_bad_lines='skip'
)

In [8]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [20]:
# TRANSFORM (Cleaning + Processing)

dc = df[['Order Date', 'Sales', 'Profit', 'Category', 'Region']]

In [ ]:
dc['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')

# Convert to MySQL format
dc['Order Date'] = dc['Order Date'].dt.strftime('%Y-%m-%d')

C:\Users\rajra\AppData\Local\Temp\ipykernel_12836\2868239509.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dc['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
C:\Users\rajra\AppData\Local\Temp\ipykernel_12836\2868239509.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dc['Order Date'] = dc['Order Date'].dt.strftime('%Y-%m-%d')


In [27]:
dc

,Order Date,Sales,Profit,Category,Region
0,2016-11-08,275.058000,41.9136,Furniture,South
1,2016-11-08,768.537000,219.5820,Furniture,South
2,2016-06-12,15.351000,6.8714,Office Supplies,West
3,2015-10-11,1005.456375,-383.0310,Furniture,South
4,2015-10-11,23.486400,2.5164,Office Supplies,South
...,...,...,...,...,...
9989,2014-01-21,26.510400,4.1028,Furniture,South
9990,2017-02-26,96.558000,15.6332,Furniture,West
9991,2017-02-26,271.504800,19.3932,Technology,West
9992,2017-02-26,31.080000,13.3200,Office Supplies,West


In [30]:
dc= dc.dropna()

In [23]:
dc['Sales'] = dc['Sales'] * 1.05

dc.head()

C:\Users\rajra\AppData\Local\Temp\ipykernel_12836\4220455574.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dc['Sales'] = dc['Sales'] * 1.05


,Order Date,Sales,Profit,Category,Region
0,2016-11-08,275.058000,41.9136,Furniture,South
1,2016-11-08,768.537000,219.5820,Furniture,South
2,2016-06-12,15.351000,6.8714,Office Supplies,West
3,2015-10-11,1005.456375,-383.0310,Furniture,South
4,2015-10-11,23.486400,2.5164,Office Supplies,South


In [24]:
# Save Cleaned Data

dc.to_csv("cleaned_superstore.csv", index=False)

In [25]:
# MySQL Connection

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="7170",
    database="etl_project"
)

cursor = conn.cursor()

In [28]:
for _, row in dc.iterrows():
    query = """
    INSERT INTO superstore_sales (order_date, sales, profit, category, region)
    VALUES (%s, %s, %s, %s, %s)
    """
    
    cursor.execute(query, (
        row['Order Date'],
        row['Sales'],
        row['Profit'],
        row['Category'],
        row['Region']
    ))

conn.commit()
print("Data Loaded Successfully")

Data Loaded Successfully


In [ ]:
# Close Connection
conn.close()